# Demo de inferencia: GPT-2, RoPE, LoRA y Full Fine-Tuning

Este notebook ejecuta las mismas frases con hasta cuatro variantes:

1. **GPT-2 original**: posiciones aprendidas y pesos oficiales.
2. **RoPE original**: pesos oficiales, descartando `wpe` (baseline RoPE sin adaptar).
3. **RoPE + LoRA**: adaptadores entrenados sobre Q/V.
4. **RoPE + Full fine-tuning**: checkpoint completo, si existe en `gpt2_weights/full/`.

Los resultados son demostrativos: RoPE original no fue entrenado con ese esquema posicional. El notebook detecta el checkpoint LoRA más reciente y, cuando exista, el full checkpoint más reciente.

## 1. Entorno y rutas

Ejecuta el notebook desde la raíz del repositorio. El modelo se cargará en GPU si CUDA está disponible.

In [1]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "minigpt").is_dir():
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
import tiktoken
import minigpt as mg
from minigpt.lora import inject_lora, load_adapter_state_dict

WEIGHTS_DIR = PROJECT_ROOT / "gpt2_weights"
LORA_RUNS = WEIGHTS_DIR / "lora"
FULL_RUNS = WEIGHTS_DIR / "full"
LORA_CHECKPOINTS = sorted(LORA_RUNS.glob("rope-gpt2-*/best.pt"), key=lambda p: p.stat().st_mtime)
FULL_CHECKPOINTS = sorted(FULL_RUNS.glob("**/best.pt"), key=lambda p: p.stat().st_mtime)
LORA_CHECKPOINT = LORA_CHECKPOINTS[-1] if LORA_CHECKPOINTS else None
FULL_CHECKPOINT = FULL_CHECKPOINTS[-1] if FULL_CHECKPOINTS else None

device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = tiktoken.get_encoding("gpt2")
print(f"minigpt {mg.__version__} | torch {torch.__version__} | dispositivo: {device}")
if device == "cuda":
    print(torch.cuda.get_device_name(0))
print("LoRA:", LORA_CHECKPOINT or "no encontrado")
print("Full FT:", FULL_CHECKPOINT or "no encontrado; se omitirá hasta entrenarlo")

minigpt 1.0.0 | torch 2.14.1+cpu | dispositivo: cpu
LoRA: c:\Users\cavir.ROG-CARLOS\Desktop\PLN-Practica\Practica1_PLN1\gpt2_weights\lora\rope-gpt2-20261018-110844\best.pt
Full FT: c:\Users\cavir.ROG-CARLOS\Desktop\PLN-Practica\Practica1_PLN1\gpt2_weights\full\rope-gpt2-20261018-115135\best.pt


## 2. Cargar las variantes

Los adaptadores se cargan sobre una instancia nueva del backbone RoPE. El checkpoint full contiene el `state_dict` completo. Si aún no existe un checkpoint full, se informará y el resto del demo continuará.

In [2]:
models = {}

# 1) GPT-2 original: learned positional embeddings + pesos oficiales.
models["GPT-2 original"] = mg.pretrained_gpt2(
    positional="learned", device=device, weights_dir=WEIGHTS_DIR
)

# 2) RoPE con los mismos pesos oficiales; wpe se descarta deliberadamente.
models["RoPE + pesos originales"] = mg.pretrained_gpt2(
    positional="rope", device=device, weights_dir=WEIGHTS_DIR
)

# 3) RoPE + adaptadores LoRA entrenados.
if LORA_CHECKPOINT is not None:
    adapter_config_path = LORA_CHECKPOINT.parent / "adapter_config.json"
    adapter_config = json.loads(adapter_config_path.read_text(encoding="utf-8"))
    rope_lora = mg.pretrained_gpt2(
        positional="rope", device=device, weights_dir=WEIGHTS_DIR
    )
    inject_lora(
        rope_lora,
        tuple(adapter_config["targets"]),
        rank=adapter_config["rank"],
        alpha=adapter_config["alpha"],
        dropout=adapter_config["dropout"],
    )
    lora_data = torch.load(LORA_CHECKPOINT, map_location="cpu", weights_only=False)
    load_adapter_state_dict(rope_lora, lora_data["adapter_state_dict"])
    rope_lora.eval()
    models["RoPE + LoRA"] = rope_lora
    print(f"LoRA cargado (paso {lora_data['step']}, val_loss={lora_data['val_loss']:.4f})")
else:
    print("No hay checkpoint LoRA; ejecuta primero el entrenamiento LoRA.")

# 4) RoPE + full fine-tuning. Se agrega automáticamente cuando exista.
if FULL_CHECKPOINT is not None:
    full_data = torch.load(FULL_CHECKPOINT, map_location="cpu", weights_only=False, mmap=True)
    full_model = mg.GPTModel(mg.GPTConfig(**full_data["model_config"]))
    full_model.load_state_dict(full_data["model_state_dict"])
    full_model.to(device).eval()
    models["RoPE + Full FT"] = full_model
    print(f"Full fine-tuning cargado (paso {full_data['step']}, val_loss={full_data['val_loss']:.4f})")
    del full_data
else:
    print("No hay checkpoint full en gpt2_weights/full/; ese modo aparecerá al entrenarlo.")

for name, model in models.items():
    total = mg.count_parameters(model, trainable_only=False)
    trainable = mg.count_parameters(model, trainable_only=True)
    print(f"{name:<26} total={mg.format_parameters(total):>14} | entrenables={mg.format_parameters(trainable):>10}")

Cargando pesos de GPT-2 desde c:\Users\cavir.ROG-CARLOS\Desktop\PLN-Practica\Practica1_PLN1\gpt2_weights ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Cargando pesos de GPT-2 desde c:\Users\cavir.ROG-CARLOS\Desktop\PLN-Practica\Practica1_PLN1\gpt2_weights ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Cargando pesos de GPT-2 desde c:\Users\cavir.ROG-CARLOS\Desktop\PLN-Practica\Practica1_PLN1\gpt2_weights ...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

LoRA cargado (paso 18400, val_loss=4.7103)
Full fine-tuning cargado (paso 20000, val_loss=3.7728)
GPT-2 original             total=   163.037.184 | entrenables=163.037.184
RoPE + pesos originales    total=   162.250.752 | entrenables=162.250.752
RoPE + LoRA                total=   162.545.664 | entrenables=   294.912
RoPE + Full FT             total=   162.250.752 | entrenables=162.250.752


## 3. Comparaci?n de calidad

Se usan las mismas frases para cada modelo. Mostramos tanto greedy (control reproducible) como muestreo con temperatura, top-k/top-p y penalizaci?n de repetici?n. Para los ejemplos OPUS, el prompt incluye la tarea expl?cita de traducci?n. Los prompts en ingl?s son un control, no una medida principal de calidad para el ajuste en espa?ol.

In [7]:
@torch.inference_mode()
def completar(model, prompt, max_new_tokens=48, modo="sampling", seed=123):
    ids = tokenizer.encode(prompt, allowed_special="all")
    input_ids = torch.tensor(ids, dtype=torch.long, device=device).unsqueeze(0)
    output = mg.generate(
        model, input_ids, max_new_tokens=max_new_tokens,
        context_size=min(512, model.cfg.context_length),
        temperature=None if modo == "greedy" else 0.8,
        top_k=None if modo == "greedy" else 50,
        top_p=None if modo == "greedy" else 0.9,
        eos_id=tokenizer.eot_token, seed=seed,
    )
    return tokenizer.decode(output[0].tolist())

def repeticion_trigramas(texto):
    tokens = tokenizer.encode(texto, allowed_special="all")
    ngrams = [tuple(tokens[i:i+3]) for i in range(max(0, len(tokens)-2))]
    return 0.0 if not ngrams else 1 - len(set(ngrams)) / len(ngrams)

In [ ]:
prompt_cases = [
    ("OPUS-100: ingles a espa?ol", "Translate English to Spanish: English: The report published this morning describes new measures to protect the environment. Spanish:"),
    ("OPUS-100: segundo ejemplo", "Translate English to Spanish: English: After a difficult season, the team announced a new plan for its supporters. Spanish:"),
    ("Espanol: tema period?stico", "Las autoridades anunciaron esta ma?ana que el nuevo plan"),
    ("Espanol: tema nuevo", "La investigaci?n publicada por la universidad concluye que"),
    ("English control", "The report published this morning said the new policy would"),
]

for label, prompt in prompt_cases:
    print("\n" + "=" * 90 + f"\n{label}\nPROMPT: {prompt}")
    for name, model in models.items():
        print(f"\n[{name} | greedy]")
        greedy_text = completar(model, prompt, modo="greedy")
        print(greedy_text)
        print(f"repetici?n de trigramas: {repeticion_trigramas(greedy_text):.1%}")
        print(f"\n[{name} | sampling]")
        sampled_text = completar(model, prompt, modo="sampling", seed=123)
        print(sampled_text)
        print(f"repetici?n de trigramas: {repeticion_trigramas(sampled_text):.1%}")


OPUS-100: ingl?s a espa?ol
PROMPT: Translate English to Spanish: English: The report published this morning describes new measures to protect the environment. Spanish:

[GPT-2 original | greedy]
Translate English to Spanish: English: The report published this morning describes new measures to protect the environment. Spanish: The report published this morning describes new measures to protect the environment.

The report published this morning describes new measures to protect the environment. Spanish: The report published this morning describes new measures to protect the environment.

The report published
repetici?n de trigramas: 60.9%

[GPT-2 original | sampling]
Translate English to Spanish: English: The report published this morning describes new measures to protect the environment. Spanish: A Spanish Government spokesperson said: "We are working to strengthen the environment by making it easier for people to access their natural resources."

A spokesman for the Department for En

## Notas

- El GPT-2 original y RoPE original usan pesos oficiales, pero solo el primero conserva `wpe`.
- LoRA usa el checkpoint más reciente encontrado bajo `gpt2_weights/lora/rope-gpt2-*/best.pt`.
- Full FT se carga desde el `best.pt` más reciente bajo `gpt2_weights/full/`. Para habilitar esa comparación, primero ejecuta `python -m minigpt.finetune --mode full ...`.
- Los modos comparan generaciones; no son una evaluación estadística de calidad.